In [2]:
import numpy as np
import uproot
import subprocess
import os
import matplotlib.pyplot as plt
import scipy.stats.qmc as qmc
from scipy.stats.qmc import Sobol
from scipy.optimize import rosen


A test for the easy_op.py file.

In [3]:
def easy_op(dim_guess,function,steps,init_size,rounds):
    for round in range(rounds):
        for dim in range(len(dim_guess)):
            test_range = np.linspace(dim_guess[dim]-init_size/pow(1.5,round),dim_guess[dim]+init_size/pow(1.5,round),steps)
            test_input = dim_guess
            result = [0]*steps
            for istep,step in enumerate(test_range):
                test_input[dim] = step
                #Here you need to input the actual function.
                global function_count
                function_count += 1
                result[istep] = float(function(test_input))
            min_index = result.index(min(result))
            dim_guess[dim] = test_range[min_index]
    return dim_guess

def easier_op(guess,function,steps,init_size,rounds,workers):
    l_bounds = [x - init_size for x in guess]
    u_bounds = [x + init_size for x in guess]

    sampler = Sobol(d=4, scramble=False)
    sample = sampler.random_base2(m=int(np.ceil(np.log2(workers))))

    qmc.scale(sample, l_bounds, u_bounds)

    best_guess = [999,999,999,999]

    for istart,start in enumerate(sample):
        guess = easy_op(start,function,steps,init_size,rounds)
        global function_count
        function_count += 1
        if function(guess) < function(best_guess):
            best_guess = guess
    
    return(best_guess)

In [7]:
global function_count
function_count = 0
result_place = easier_op([-1,0,1,0],rosen,5,0.5,4,32)
result_value = rosen(result_place)
print(function_count*0.25)
print(result_place)
error = 0
for i in result_place:
    error += pow((i-1),2)
print(error)


648.0
[0.97569444 0.94791667 0.90625    0.84837963]
0.035081232853223634


In [8]:
def beam_deviance(scales):
    os.environ["SCALE15"] = scales[0]
    os.environ["SCALE16"] = scales[1]
    os.environ["SCALE17"] = scales[2]
    os.environ["SCALE18"] = scales[3]
    subprocess.run("g4bl $PIM1/piM1_mu.g4bl", shell=True, check=True)
    with uproot.open("piM1_plastic_decay_bend_air_mu+_155_1.0_300_12349.root") as file:
        #get the standard deviation in x and y at the end
        branch_name = 'NTuple/22150'
        x_vals = file[branch_name]['x'].array(library="np")
        y_vals = file[branch_name]['y'].array(library="np")
        x_std = np.std(x_vals)
        y_std = np.std(y_vals)
    os.remove("piM1_plastic_decay_bend_air_mu+_155_1.0_300_12349.root")
    return np.sqrt(x_std**2+y_std**2)

'''
#Arbitrary bounds; if the minimum is found to be very close to 4 or 0.001 for any of them, I will change them.
bounds = [(1,3),(1,3),(1,3),(1,3)]

results = dict()
results['DE'] = optimize.differential_evolution(beam_deviance,bounds) #Add maxiter = n if this takes too long
print(results['DE'])
'''

"\n#Arbitrary bounds; if the minimum is found to be very close to 4 or 0.001 for any of them, I will change them.\nbounds = [(1,3),(1,3),(1,3),(1,3)]\n\nresults = dict()\nresults['DE'] = optimize.differential_evolution(beam_deviance,bounds) #Add maxiter = n if this takes too long\nprint(results['DE'])\n"

In [ ]:
easier_op([1,1,1,1],beam_deviance,4,1,4,32)

In [3]:
with uproot.open("piM1_plastic_decay_bend_air_mu+_155_1.0_300_12349.root") as file:
    print(file.keys())

['NTuples;1', 'NTuples/TimeStep;1', 'VirtualDetector;1', 'VirtualDetector/InitDet;1', 'VirtualDetector/S_QTA11;1', 'VirtualDetector/M_QTA1_QTB1;1', 'VirtualDetector/E_AQTB1;1', 'VirtualDetector/E_QTB2;1', 'VirtualDetector/Det1;1', 'VirtualDetector/E_ASM1;1', 'VirtualDetector/C_IFP;1']
